# Manufacturing Defect Detection MVP - Colab Training

**GPU-Accelerated Model Training on Google Colab**

Train a ResNet18 defect detection model in ~30 minutes on free GPU.

## What This Notebook Does
1. Setup GPU environment
2. Create project structure
3. Generate source code (model, data loader)
4. Create/load MVTec AD dataset
5. Train model with validation
6. Evaluate on test set
7. Download trained checkpoint

## Expected Timeline
- Setup: 3 minutes
- Data prep: 2 minutes
- Training (20 epochs): 20 minutes
- **Total: ~30 minutes**

## Next Steps
After training:
1. Download `best_model.pt`
2. Copy to local project: `checkpoints/best_model.pt`
3. Run: `streamlit run app.py`
4. Upload images to test defect detection!

## Step 1: Check GPU Availability

In [ ]:
import torch

print('=' * 60)
print('GPU AVAILABILITY')
print('=' * 60)
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    device = 'cuda'
else:
    print('WARNING: GPU not available, will train on CPU (very slow!)')
    print('To enable GPU: Runtime > Change runtime type > GPU')
    device = 'cpu'

print(f'\nUsing device: {device.upper()}')

## Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision scikit-learn tqdm pillow numpy matplotlib scikit-image

## Step 3: Setup Project Structure

In [ ]:
import os
import sys

# Create project structure
os.makedirs('defect_detector/src', exist_ok=True)
os.makedirs('defect_detector/data', exist_ok=True)
os.makedirs('defect_detector/checkpoints', exist_ok=True)
os.chdir('defect_detector')

print(f'Working directory: {os.getcwd()}')
print('Project structure created:')
for d in ['src', 'data', 'checkpoints']:
    print(f'  - {d}/')

## Step 4: Create Source Code (model.py)

In [ ]:
model_py_code = r'''
import torch
import torch.nn as nn
from torchvision import models

class DefectDetectionModel(nn.Module):
    """ResNet18-based binary classifier for defect detection"""
    
    def __init__(self, num_classes=2, pretrained=True, dropout_rate=0.5):
        super(DefectDetectionModel, self).__init__()
        # Pretrained ResNet18 backbone
        self.backbone = models.resnet18(pretrained=pretrained)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()  # Remove original classification head
        
        # Custom classification head
        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )
        self.num_classes = num_classes
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)
        return logits
    
    def freeze_backbone(self):
        """Freeze backbone for transfer learning"""
        for param in self.backbone.parameters():
            param.requires_grad = False
    
    def unfreeze_backbone(self):
        """Unfreeze backbone for fine-tuning"""
        for param in self.backbone.parameters():
            param.requires_grad = True
'''.strip()

with open('src/model.py', 'w') as f:
    f.write(model_py_code)

print('✓ Created src/model.py (ResNet18 model)')

## Step 5: Create Source Code (data_loader.py)

In [ ]:
data_loader_code = r'''
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

class MVTecADDataset(Dataset):
    """PyTorch Dataset for MVTec AD anomaly detection"""
    
    def __init__(self, image_paths, labels, transform=None, img_size=(224, 224)):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.img_size = img_size
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        image = image.resize(self.img_size, Image.Resampling.LANCZOS)
        
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
            image = transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )(image)
        
        return image, label

class DataManager:
    """Manages MVTec AD dataset loading and preprocessing"""
    
    def __init__(self, data_dir='data', img_size=(224, 224)):
        self.data_dir = data_dir
        self.img_size = img_size
    
    def load_category(self, category):
        """Load images and labels for a category"""
        category_dir = os.path.join(self.data_dir, category)
        if not os.path.exists(category_dir):
            raise FileNotFoundError(f"Category {category} not found")
        
        image_paths = []
        labels = []
        
        # Load normal images (label=0)
        normal_dir = os.path.join(category_dir, 'train', 'good')
        if os.path.exists(normal_dir):
            for fname in os.listdir(normal_dir):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(normal_dir, fname))
                    labels.append(0)
        
        # Load defective images (label=1)
        test_dir = os.path.join(category_dir, 'test')
        if os.path.exists(test_dir):
            for defect_type in os.listdir(test_dir):
                defect_path = os.path.join(test_dir, defect_type)
                if os.path.isdir(defect_path) and defect_type != 'good':
                    for fname in os.listdir(defect_path):
                        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                            image_paths.append(os.path.join(defect_path, fname))
                            labels.append(1)
        
        return image_paths, labels
    
    def split_data(self, image_paths, labels, val_ratio=0.1, test_ratio=0.1, random_state=42):
        """Split data into train, val, test"""
        train_paths, temp_paths, train_labels, temp_labels = train_test_split(
            image_paths, labels,
            test_size=val_ratio+test_ratio,
            random_state=random_state,
            stratify=labels
        )
        
        val_size = val_ratio / (val_ratio + test_ratio)
        val_paths, test_paths, val_labels, test_labels = train_test_split(
            temp_paths, temp_labels,
            test_size=1-val_size,
            random_state=random_state,
            stratify=temp_labels
        )
        
        return {
            'train': (train_paths, train_labels),
            'val': (val_paths, val_labels),
            'test': (test_paths, test_labels)
        }
    
    def get_dataloaders(self, category, batch_size=32, num_workers=0, augment=True):
        """Create dataloaders for train/val/test"""
        image_paths, labels = self.load_category(category)
        splits = self.split_data(image_paths, labels)
        
        # Data augmentation for training
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]) if augment else transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # No augmentation for validation/test
        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        dataloaders = {}
        for split_name in ['train', 'val', 'test']:
            paths, split_labels = splits[split_name]
            dataset = MVTecADDataset(
                paths, split_labels,
                train_transform if split_name == 'train' else test_transform,
                self.img_size
            )
            dataloaders[split_name] = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=(split_name == 'train'),
                num_workers=num_workers
            )
        
        return dataloaders
'''.strip()

with open('src/data_loader.py', 'w') as f:
    f.write(data_loader_code)

print('✓ Created src/data_loader.py (dataset management)')

## Step 6: Create Mock Dataset (for quick testing)

In [ ]:
import numpy as np
from PIL import Image

def create_mock_mvtec_dataset():
    """Create a small mock dataset for testing"""
    category = 'bottle'
    
    # Create directory structure
    os.makedirs(f'data/{category}/train/good', exist_ok=True)
    os.makedirs(f'data/{category}/test/crack', exist_ok=True)
    os.makedirs(f'data/{category}/test/contamination', exist_ok=True)
    
    # Create mock images (random tensors)
    # Normal/good images (lighter colors)
    for i in range(25):
        img_array = np.random.randint(100, 200, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        img.save(f'data/{category}/train/good/normal_{i:03d}.png')
    
    # Defective images - crack type (darker colors)
    for i in range(10):
        img_array = np.random.randint(50, 100, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        img.save(f'data/{category}/test/crack/crack_{i:03d}.png')
    
    # Defective images - contamination type (bright colors)
    for i in range(10):
        img_array = np.random.randint(150, 255, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        img.save(f'data/{category}/test/contamination/contam_{i:03d}.png')
    
    print(f'✓ Mock dataset created (bottle category)')
    print(f'  - Normal: 25 images')
    print(f'  - Defective: 20 images (10 crack + 10 contamination)')

create_mock_mvtec_dataset()

## Step 7: Load Data and Create Model

In [ ]:
sys.path.insert(0, 'src')

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from model import DefectDetectionModel
from data_loader import DataManager

# Training configuration
CATEGORY = 'bottle'
NUM_EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('=' * 60)
print('TRAINING CONFIGURATION')
print('=' * 60)
print(f'Category: {CATEGORY}')
print(f'Epochs: {NUM_EPOCHS}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Learning Rate: {LEARNING_RATE}')
print(f'Device: {DEVICE.upper()}')
print()

# Load dataset
print('Loading dataset...')
data_manager = DataManager(data_dir='data')
dataloaders = data_manager.get_dataloaders(CATEGORY, batch_size=BATCH_SIZE, augment=True)

print(f'  Train batches: {len(dataloaders["train"])}')
print(f'  Val batches: {len(dataloaders["val"])}')
print(f'  Test batches: {len(dataloaders["test"])}')
print()

# Create model
print('Creating model...')
model = DefectDetectionModel(num_classes=2, pretrained=True)
model = model.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total parameters: {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}')
print()
print('✓ Model and data ready for training')

## Step 8: Train Model

In [ ]:
# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=False
)

# Freeze backbone for transfer learning
model.freeze_backbone()
best_val_accuracy = 0.0

print('=' * 60)
print('TRAINING STARTED')
print('=' * 60)
print()

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for images, labels in tqdm(dataloaders['train'], desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]', leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(dataloaders['train'])
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloaders['val'], desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Val]', leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_loss /= len(dataloaders['val'])
    val_accuracy = 100.0 * correct / total
    
    # Print progress
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_accuracy:.2f}%')
    
    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), 'checkpoints/best_model.pt')
        print(f'           -> Saved best model (accuracy: {val_accuracy:.2f}%)')
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Unfreeze backbone after a few epochs for fine-tuning
    if epoch == 5:
        model.unfreeze_backbone()
        print('           -> Unfroze backbone for fine-tuning')

print()
print('=' * 60)
print(f'TRAINING COMPLETE')
print(f'Best Validation Accuracy: {best_val_accuracy:.2f}%')
print('=' * 60)

## Step 9: Evaluate on Test Set

In [ ]:
# Test set evaluation
print('\nEvaluating on test set...')
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(dataloaders['test'], desc='Testing', leave=False):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss /= len(dataloaders['test'])
test_accuracy = 100.0 * correct / total

print(f'\nTest Results:')
print(f'  Test Loss: {test_loss:.4f}')
print(f'  Test Accuracy: {test_accuracy:.2f}%')
print(f'\n✓ Model checkpoint saved: checkpoints/best_model.pt')

## Step 10: Download Checkpoint

In [ ]:
from google.colab import files

print('Downloading trained model checkpoint...')
files.download('checkpoints/best_model.pt')
print('\n✓ Downloaded: best_model.pt')
print('\nNEXT STEPS:')
print('1. Save the downloaded file as: checkpoints/best_model.pt (in your local project)')
print('2. Run the web app: streamlit run app.py')
print('3. Upload images to detect defects!')

## Done! 🎉

### Summary
- ✅ Model trained with transfer learning
- ✅ Best accuracy: **see above**
- ✅ Checkpoint downloaded

### What's Next
1. **Local Setup**: Install dependencies locally (`pip install -r requirements.txt`)
2. **Move Checkpoint**: Copy `best_model.pt` to `checkpoints/best_model.pt`
3. **Run Web App**: `streamlit run app.py`
4. **Test**: Upload images and see defect detection + Grad-CAM heatmaps!

### To Train on Real MVTec AD Dataset
1. Download from: https://www.mvtec.com/company/research/datasets/mvtec-ad
2. Upload to Colab
3. Extract to `data/` folder
4. Change `CATEGORY = 'bottle'` to another category
5. Re-run training cells
6. Download new checkpoint

### Training on Other Categories
Replace `'bottle'` with any of:
- cable, carpet, grid, hazelnut, leather, metal_nut, pill, screw
- tile, toothbrush, transistor, wood, zipper